In [ ]:
!git clone https://github.com/elifsudayi/food-recipe-finder.git
%cd food-recipe-finder
!pip install -q -r requirements.txt
!mkdir -p results models

import sys, json, time; sys.path.append(".")
import numpy as np, pandas as pd, torch
from src.utils import get_device, hardware_info, model_size_mb, count_params

corpus = pd.read_csv("corpus/recipes.csv"); queries = pd.read_csv("corpus/queries.csv")
corpus["description"] = corpus["description"].fillna(""); corpus["diet_tags"] = corpus["diet_tags"].fillna("")
device = get_device(); print(device, corpus.shape, queries.shape)

Cloning into 'food-recipe-finder'...
remote: Enumerating objects: 73, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 73 (delta 27), reused 59 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (73/73), 171.45 KiB | 5.91 MiB/s, done.
Resolving deltas: 100% (27/27), done.
/content/food-recipe-finder
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
cuda (300, 8) (40, 6)


In [ ]:
from src.retrieval.embed import STEncoder, EmbeddingRetriever
from src.retrieval.tfidf import doc_text

enc = STEncoder(device=str(device))
print(enc.model)
print("max_seq_length:", enc.model.max_seq_length, "| parameter:", count_params(enc.model))

docs = [doc_text(r) for _, r in corpus.iterrows()]
lens = np.array([len(enc.model.tokenizer(d)["input_ids"]) for d in docs])
print("token length percentiles (50/90/99/100):", np.percentile(lens, [50, 90, 99, 100]))
print("rate of truncated sequences:", float((lens > enc.model.max_seq_length).mean()))

ret_c = EmbeddingRetriever(corpus, enc, use_constraints=True)
ret_n = EmbeddingRetriever(corpus, enc, use_constraints=False, doc_emb=ret_c.E)
print("document embedding format:", ret_c.E.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (323 > 256). Running this sequence through the model will result in indexing errors


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)
max_seq_length: 256 | parameter: 22713216
token length percentiles (50/90/99/100): [ 82.  155.1 264.  323. ]
rate of truncated sequences: 0.016666666666666666
document embedding format: (300, 384)


In [ ]:
for q in ["raw fish and seasoned rice rolled in seaweed",
          "cold creamy frozen dessert to cool down on a hot day",
          "vegan french fries under 45 minutes"]:
    top, sc, c = ret_c.search(q, 3)
    print(f"\n{q!r}  | constraint: {c.max_minutes} dk, {c.diets}")
    for i, s in zip(top, sc):
        r = ret_c.corpus.iloc[i]; print(f"   {s:.3f}  {r['title']}  [{r['dish']}, {r['minutes']} minutes, {r['diet_tags']}]")


'raw fish and seasoned rice rolled in seaweed'  | constraint: None dk, []
   0.566  Fusion Chicken Potato Sushi Rolls  [sushi, 25 minutes, ]
   0.558  Cucumber And Wasabi Sushi  [sushi, 15 minutes, low-fat|low-sodium|vegan|vegetarian]
   0.549  Spicy Tuna Roll Sushi  [sushi, 20 minutes, ]

'cold creamy frozen dessert to cool down on a hot day'  | constraint: None dk, []
   0.593  Homemade Strawberry Ice Cream  [ice_cream, 330 minutes, low-sodium]
   0.585  Coffee Filbert Ice Cream  [ice_cream, 150 minutes, low-sodium]
   0.584  Frozen Fruit Yogurt For Ice Cream Machine  [ice_cream, 5 minutes, ]

'vegan french fries under 45 minutes'  | constraint: 45.0 dk, ['vegan']
   0.638  Easy Fat Free Seasoned French Fries  [french_fries, 25 minutes, vegan|vegetarian]
   0.528  Sweet Potato Idaho Fries  [french_fries, 40 minutes, low-sodium|vegan|vegetarian]
   0.483  Ina Garten S Baked Sweet Potato Fries  [french_fries, 40 minutes, vegan|vegetarian]


In [ ]:
from src.retrieval.tfidf import TfidfRetriever
from src.retrieval.metrics import evaluate

systems = {"tf-idf stem + constraints (Step 7 baseline)": TfidfRetriever(corpus, "stem", True).rank_ids,
           "MiniLM, no constraints": ret_n.rank_ids,
           "MiniLM + constraints [TRANSFORMER]": ret_c.rank_ids}
res, perq = {}, {}
for name, fn in systems.items():
    s, by, pq = evaluate(fn, queries)
    res[name] = {"summary": s, "by_type": by.round(4).to_dict()}; perq[name] = pq
    print(f"{name:46s} MRR {s['MRR']:.3f} | Recall@5 {s['Recall@5']:.3f}")

cmp = pd.concat({n: pd.DataFrame(v["by_type"])[["MRR", "recall@5"]] for n, v in res.items()}, axis=1)
print("\nBy query type:\n", cmp.round(3).to_string())

tf-idf stem + constraints (Step 7 baseline)    MRR 0.946 | Recall@5 0.855
MiniLM, no constraints                         MRR 0.677 | Recall@5 0.571
MiniLM + constraints [TRANSFORMER]             MRR 0.938 | Recall@5 0.883

By query type:
            tf-idf stem + constraints (Step 7 baseline)          MiniLM, no constraints          MiniLM + constraints [TRANSFORMER]         
                                                   MRR recall@5                    MRR recall@5                                MRR recall@5
combo                                            1.000    1.000                  0.531    0.333                              1.000    0.938
diet                                             1.000    0.950                  0.376    0.242                              1.000    0.975
ingredient                                       0.900    0.740                  0.803    0.657                              0.803    0.657
semantic                                         0.883    0.7

In [ ]:
T = "MiniLM + constraints [TRANSFORMER]"; B = "tf-idf stem + constraints (Step 7 baseline)"
d = perq[T][["query_id", "type", "query", "rr", "recall@5"]].merge(
    perq[B][["query_id", "rr", "recall@5"]], on="query_id", suffixes=("_mini", "_tfidf"))
print("Queries for which MiniLM performs worse than tf-idf:")
print(d[d["rr_mini"] < d["rr_tfidf"]].to_string())
print("\nQueries for which MiniLM's Recall@5 < 1:")
print(d[d["recall@5_mini"] < 1][["query_id", "type", "query", "rr_mini", "recall@5_mini"]].to_string())

Queries for which MiniLM performs worse than tf-idf:
   query_id        type                      query   rr_mini  recall@5_mini  rr_tfidf  recall@5_tfidf
16      q17  ingredient       pizza with mushrooms  0.333333            0.4       1.0             0.6
20      q21  ingredient  caesar salad with chicken  0.500000            0.8       1.0             0.6
24      q25  ingredient   french fries with garlic  0.200000            0.2       0.5             0.2

Queries for which MiniLM's Recall@5 < 1:
   query_id        type                                                           query   rr_mini  recall@5_mini
0       q01        time                                    quick pizza under 20 minutes  1.000000       0.800000
9       q10        diet                                                vegetarian pizza  1.000000       0.800000
16      q17  ingredient                                            pizza with mushrooms  0.333333       0.400000
17      q18  ingredient                      

In [ ]:
enc_cpu = STEncoder(device="cpu")
ret_cpu = EmbeddingRetriever(corpus, enc_cpu, True, doc_emb=ret_c.E)

qs = queries["query"].tolist() * 5                    # 200 calls
for q in qs[:10]: ret_cpu.search(q, 5)                # warm-up
t_search, t_enc = [], []
for q in qs:
    t0 = time.perf_counter(); ret_cpu.search(q, 5); t_search.append((time.perf_counter() - t0) * 1000)
    t0 = time.perf_counter(); enc_cpu.encode([q]);   t_enc.append((time.perf_counter() - t0) * 1000)

out = {"model": "all-MiniLM-L6-v2_fp32 + constraints", "summary": res[T]["summary"], "by_type": res[T]["by_type"],
       "size_mb": model_size_mb(enc_cpu.model), "params": count_params(enc_cpu.model),
       "latency_cpu_search_ms": {"mean_ms": float(np.mean(t_search)), "std_ms": float(np.std(t_search)), "runs": len(t_search)},
       "latency_cpu_encode_query_ms": {"mean_ms": float(np.mean(t_enc)), "std_ms": float(np.std(t_enc)), "runs": len(t_enc)},
       "ablation": {k: v["summary"] for k, v in res.items()},
       "truncated_doc_fraction": float((lens > enc.model.max_seq_length).mean()),
       "hardware": hardware_info()}
json.dump(out, open("results/retrieval_minilm_metrics.json", "w"), indent=2)
np.save("models/doc_emb_minilm_fp32.npy", ret_c.E)
print(json.dumps(out, indent=2))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

{
  "model": "all-MiniLM-L6-v2_fp32 + constraints",
  "summary": {
    "MRR": 0.9383333333333332,
    "Recall@5": 0.8829166666666666,
    "n_queries": 40
  },
  "by_type": {
    "MRR": {
      "combo": 1.0,
      "diet": 1.0,
      "ingredient": 0.8033,
      "semantic": 0.95,
      "time": 1.0
    },
    "recall@5": {
      "combo": 0.9375,
      "diet": 0.975,
      "ingredient": 0.6567,
      "semantic": 0.94,
      "time": 0.975
    }
  },
  "size_mb": 90.893873,
  "params": 22713216,
  "latency_cpu_search_ms": {
    "mean_ms": 17.145980469990718,
    "std_ms": 6.857400798888076,
    "runs": 200
  },
  "latency_cpu_encode_query_ms": {
    "mean_ms": 18.0620503599971,
    "std_ms": 5.958413318954986,
    "runs": 200
  },
  "ablation": {
    "tf-idf stem + constraints (Step 7 baseline)": {
      "MRR": 0.9458333333333334,
      "Recall@5": 0.8550000000000001,
      "n_queries": 40
    },
    "MiniLM, no constraints": {
      "MRR": 0.6770955882352941,
      "Recall@5": 0.570833333333